## Classroom Maker I GREEN 2025

In [ ]:
import os.path
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import gspread
from google.oauth2 import service_account

# ==============================
# CONFIGURACIÓN DE CREDENCIALES
# ==============================
SCOPES = [
    "https://www.googleapis.com/auth/classroom.courses",
    "https://www.googleapis.com/auth/classroom.rosters",
    "https://www.googleapis.com/auth/spreadsheets.readonly",
]

# Configuración para autenticación OAuth
CLIENT_SECRET_FILE = "credentials.json"
TOKEN_FILE = "token.json"


# Autenticación con OAuth 2.0
def get_oauth_credentials():
    creds = None
    if os.path.exists(TOKEN_FILE):
        creds = Credentials.from_authorized_user_file(TOKEN_FILE, SCOPES)

    # Si no hay credenciales (o no son válidas), solicitamos autorización
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(CLIENT_SECRET_FILE, SCOPES)
            creds = flow.run_local_server(port=0)
        # Guardar las credenciales para la próxima ejecución
        with open(TOKEN_FILE, "w") as token:
            token.write(creds.to_json())

    return creds


# Obtener credenciales para Google Classroom
credentials = get_oauth_credentials()

# Crear clientes para las APIs
classroom_service = build("classroom", "v1", credentials=credentials)

# Para Google Sheets, usamos gspread con OAuth
gc = gspread.oauth(
    credentials_filename=CLIENT_SECRET_FILE, authorized_user_filename=TOKEN_FILE
)

# ==============================
# LECTURA DE DATOS DE GOOGLE SHEETS
# ==============================
SPREADSHEET_ID = "1dMXhr58YylEb0F7SbKZKGVHF1VMKaLvH0lxO4t4bMmQ"
try:
    spreadsheet = gc.open_by_key(SPREADSHEET_ID)
    sheet_cursos = spreadsheet.worksheet("Asignaturas")
    sheet_alumnos = spreadsheet.worksheet("Curso")

    cursos = sheet_cursos.get_all_values()[1:]  # Saltar encabezados
    alumnos = sheet_alumnos.get_all_values()[1:]  # Saltar encabezados
except Exception as e:
    print(f"Error al acceder a Google Sheets: {e}")
    exit()

# Definir índices de columnas
COLS = {
    "CourseName": 0,
    "ClassName": 1,
    "MailAssociateTeacher": 3,
    "Section": 4,
    "Description": 5,
    "Room": 6,
    "CourseID": 7,
    "MailAssistantTeacher": 8,
}

# ==============================
# CREACIÓN DE CURSOS EN CLASSROOM
# ==============================


def create_students(course_id, group):
    """Añadir estudiantes al curso en Google Classroom."""
    for alumno in alumnos:
        student_email = alumno[2]  # Columna con el correo del estudiante
        student_data = {"userId": student_email}
        try:
            classroom_service.courses().students().create(
                courseId=course_id, body=student_data
            ).execute()
            print(f"Estudiante {student_email} añadido al curso {course_id}")
        except Exception as e:
            print(f"Error añadiendo {student_email}: {e}")


def create_courses():
    """Crear cursos en Google Classroom y asignar profesores."""
    for fila, curso in enumerate(cursos):
        course_data = {
            "name": curso[COLS["CourseName"]],
            "section": curso[COLS["Section"]],
            "descriptionHeading": curso[COLS["Description"]],
            "room": curso[COLS["Room"]],
            "ownerId": "me",  # "me" se refiere al usuario autenticado
            "courseState": "ACTIVE",
        }

        try:
            course = classroom_service.courses().create(body=course_data).execute()
            course_id = course["id"]
            print(f"Curso creado: {course['name']} (ID: {course_id})")

            # Añadir profesor principal si es diferente del usuario autenticado
            if curso[COLS["MailAssociateTeacher"]]:
                teacher_data = {"userId": curso[COLS["MailAssociateTeacher"]]}
                try:
                    classroom_service.courses().teachers().create(
                        courseId=course_id, body=teacher_data
                    ).execute()
                    print(
                        f"Profesor principal {curso[COLS['MailAssociateTeacher']]} asignado"
                    )
                except Exception as e:
                    print(f"Error asignando profesor principal: {e}")

            # Añadir profesor asistente
            if curso[COLS["MailAssistantTeacher"]]:
                teacher_data = {"userId": curso[COLS["MailAssistantTeacher"]]}
                try:
                    classroom_service.courses().teachers().create(
                        courseId=course_id, body=teacher_data
                    ).execute()
                    print(
                        f"Profesor asistente {curso[COLS['MailAssistantTeacher']]} asignado"
                    )
                except Exception as e:
                    print(f"Error asignando profesor asistente: {e}")

            # Actualizar ID del curso en la hoja
            sheet_cursos.update_cell(fila + 2, COLS["CourseID"] + 1, course_id)

            # Añadir estudiantes
            create_students(course_id, curso[COLS["ClassName"]])

        except Exception as e:
            print(f"Error creando curso {curso[COLS['CourseName']]}: {e}")


# Ejecutar la creación de cursos
if __name__ == "__main__":
    create_courses()

Error al acceder a Google Sheets: 
Error creando curso FISIC - I° G - 2025: <HttpError 403 when requesting https://classroom.googleapis.com/v1/courses?alt=json returned "Request had insufficient authentication scopes.". Details: "[{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'ACCESS_TOKEN_SCOPE_INSUFFICIENT', 'domain': 'googleapis.com', 'metadata': {'service': 'classroom.googleapis.com', 'method': 'google.classroom.v1.Courses.CreateCourse'}}]">
Error creando curso BIOL - I° G - 2025: <HttpError 403 when requesting https://classroom.googleapis.com/v1/courses?alt=json returned "Request had insufficient authentication scopes.". Details: "[{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'ACCESS_TOKEN_SCOPE_INSUFFICIENT', 'domain': 'googleapis.com', 'metadata': {'service': 'classroom.googleapis.com', 'method': 'google.classroom.v1.Courses.CreateCourse'}}]">
Error creando curso TECNO - I° G - 2025: <HttpError 403 when requesting https://classroom.googleapis

: 

In [8]:
import os.path

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# If modifying these scopes, delete the file token.json.
SCOPES = ["https://www.googleapis.com/auth/classroom.courses.readonly"]


def main():
    """Shows basic usage of the Classroom API.
    Prints the names of the first 10 courses the user has access to.
    """
    creds = None
    # The file token.json stores the user's access and refresh tokens, and is
    # created automatically when the authorization flow completes for the first
    # time.
    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)
    # If there are no (valid) credentials available, let the user log in.
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file("credentials.json", SCOPES)
            creds = flow.run_local_server(port=0)
        # Save the credentials for the next run
        with open("token.json", "w") as token:
            token.write(creds.to_json())

    try:
        service = build("classroom", "v1", credentials=creds)

        # Call the Classroom API
        results = service.courses().list(pageSize=10).execute()
        courses = results.get("courses", [])

        if not courses:
            print("No courses found.")
            return
        # Prints the names of the first 10 courses.
        print("Courses:")
        for course in courses:
            print(course["name"])

    except HttpError as error:
        print(f"An error occurred: {error}")


if __name__ == "__main__":
    main()

Courses:
F.VALOR - 7° A - 2025
MUSIC - 7° A - 2025
ARTES - 7° A - 2025
TECNO - 7° A - 2025
F.VALOR - 8° A - 2025
BIOL - 7° A - 2025
QUIM - 8° A - 2025
FISIC - 7° A - 2025
MATE - 8° A - 2025
INGLES - 8° A - 2025
